# 06 — CART Baseline

**Section:** Evaluation layer (PROMETHEE II, CART baseline, statistical testing, GSEA)
**Depends on:** `01_synthetic_data_foundation.ipynb`, `03_promethee_ranking.ipynb`
**Feeds into:** re-run of `03`'s unified PROMETHEE (evolutionary vs CART), `04`'s statistical
testing pattern extended to a third comparison

## Recommended reading before this notebook

- **Breiman et al., *Classification and Regression Trees*** — CART itself.
- **CONTRACTS.md Arm 5 / methodology §4.6.4** — "CART baseline (`scikit-learn`, depth-bounded)
  fitted to each model's predictions, rules extracted and pushed through the *same*
  PROMETHEE run for a SOTA comparison." The key word is **same**: CART isn't a competing
  predictor being benchmarked against DeepCCDS, it's a second, more conventional rule
  extractor benchmarked against GA/NSGA-II, all judged by the identical `metrics.py`
  functions and the identical unified PROMETHEE run.
- **Orientation doc, Part F2** — "is the CART baseline a required deliverable or a
  time-permitting extension" is still an open question for supervisors; this notebook exists
  either way so the answer doesn't gate whether the code is ready.

## What this notebook does

Fits a depth-bounded `DecisionTreeClassifier` per condition (baseline, SAM) on the same
discretised features GA/NSGA-II use, walks the tree's root-to-leaf paths to extract IF-THEN
rules in the **same §0.4 schema** as GA/NSGA-II, scores them with the same shared
`metrics.py` functions, and writes a `cart_{baseline,sam}.json` rule pool that validates
against `schema.py` and can be loaded into notebook 03's unified PROMETHEE run alongside the
other four pools.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, "..")  # repo root, so `import src...` resolves

from src.common import metrics, schema
from src.common.config import load_config
from src.extraction.common import resolve_genes

cfg = load_config("../configs/pipeline.yaml")
print(f"config_hash: {cfg.config_hash}")

FIXTURE_DIR = Path("../tests/fixtures/synthetic")
RULES_DIR = FIXTURE_DIR / "rules"
feature_map = json.load(open(FIXTURE_DIR / "feature_map.json"))


config_hash: a2070aedcc19


## 1. A schema gap, found by running, not reading

`schema.py`'s `VALID_EXTRACTORS` was `{"ga", "nsga"}` — no `"cart"`. Confirmed directly:
`validate_rule_pool({"meta": {"extractor": "cart", ...}, "rules": []})` returned a validation
error before any rule content was even considered. This is exactly the kind of gap CONTRACTS.md
§6 exists to catch ("if you change a schema... that's a breaking change; announce it").

**Fixed in `src/common/schema.py`** (this session, same repo this notebook runs against):
`VALID_EXTRACTORS = {"ga", "nsga", "cart"}`, with a comment explaining why and dating the
change. **This needs a team announcement** — flagging here rather than silently shipping it,
per the working agreement. It's additive and backward-compatible (existing `ga`/`nsga` pools
are unaffected), but it's still a shared-code change.


In [2]:
from src.common.schema import VALID_EXTRACTORS, validate_rule_pool
assert "cart" in VALID_EXTRACTORS, "expected schema.py's VALID_EXTRACTORS fix to already be applied"
print("VALID_EXTRACTORS:", VALID_EXTRACTORS)


VALID_EXTRACTORS: {'cart', 'nsga', 'ga'}


## 2. Rebuild the discretised feature matrix

Same discretisation as notebooks 02/03 (z-score thresholds from `pipeline.yaml` for pathway
activity, binary mutation status mapped to Low/High) — CART needs to see the **same**
discretised feature space GA/NSGA-II operate over, since the whole point of the comparison is
"does a conventional tree-based extractor find equally good rules over the identical feature
representation," not "does CART benefit from finer-grained continuous inputs."


In [3]:
features_df = pd.read_csv(FIXTURE_DIR / "synthetic_features.csv", index_col="cell_line_id")
labels_df = pd.read_csv(FIXTURE_DIR / "synthetic_labels.csv", index_col="cell_line_id")

def discretise_pathway(series: pd.Series, high_z: float, low_z: float) -> pd.Series:
    z = (series - series.mean()) / series.std()
    return pd.Series(
        np.select([z > high_z, z < low_z], ["High", "Low"], default="Intermediate"),
        index=series.index,
    )

def discretise_mutation(series: pd.Series) -> pd.Series:
    return series.map({0: "Low", 1: "High"})

gene_cols = [c for c in features_df.columns if c.startswith("GENE_")]
pathway_cols = [c for c in features_df.columns if c.startswith("PATHWAY_")]
feature_cols = gene_cols + pathway_cols

discretised = features_df[feature_cols].copy()
for c in gene_cols:
    discretised[c] = discretise_mutation(features_df[c])
for c in pathway_cols:
    discretised[c] = discretise_pathway(features_df[c], cfg.discretisation.high_z, cfg.discretisation.low_z)

# CART itself needs numeric input -- one-hot encode each feature's partition labels so a tree
# split on a single column corresponds directly to one rule condition ("feature == partition").
onehot = pd.get_dummies(discretised[feature_cols], columns=feature_cols)

discretised = discretised.join(labels_df[["cancer_type", "baseline_pred_class", "sam_pred_class"]])
CLASS_COL = {"baseline": "baseline_pred_class", "sam": "sam_pred_class"}

print(f"discretised: {discretised.shape}, one-hot design matrix: {onehot.shape}")


discretised: (300, 61), one-hot design matrix: (300, 154)


## 3. Extracting IF-THEN rules from tree paths

A CART split is a binary test `one_hot_column > 0.5`, i.e. "is this feature exactly this
partition." The **left** branch ("not this partition") isn't representable as a positive
`{feature, partition}` condition in the §0.4 schema, so this extraction only keeps **right**
(positive) branches along each root-to-leaf path and drops the negative ones. A leaf reached
only through negative branches for some feature simply doesn't mention that feature in its
rule — the rule is broader than the leaf's literal sample set, which is exactly why coverage,
fidelity, and the predicted class are all **recomputed from scratch** against the full
discretised dataset using the same `metrics.evaluate_rule_mask()` / `rule_class_preds()` /
`fidelity_f1()` functions GA and NSGA-II use, rather than trusted from the tree's own leaf
statistics. Same 2% coverage floor (C4/C6) applied for consistency with the other extractors.

`max_depth=4` keeps rules short enough to stay comprehensible (Rule Explosion, §3.6.1) — an
arbitrary but defensible starting point, flagged as adjustable.


In [4]:
MIN_COVERAGE = 0.02  # C4/C6, same threshold as GA/NSGA-II
MAX_DEPTH = 4          # arbitrary but comprehensibility-motivated -- flagged as adjustable
MIN_SAMPLES_LEAF = 5


def extract_cart_rules(condition: str, class_col: str, seed: int) -> list[dict]:
    y = discretised[class_col]
    clf = DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=seed)
    clf.fit(onehot.to_numpy(), y.to_numpy())

    tree = clf.tree_
    onehot_cols = list(onehot.columns)

    leaf_paths = []

    def recurse(node, path):
        is_leaf = tree.children_left[node] == tree.children_right[node]
        if is_leaf:
            leaf_paths.append(list(path))
            return
        col = onehot_cols[tree.feature[node]]
        feature, partition = col.rsplit("_", 1)  # e.g. "PATHWAY_19_High" -> ("PATHWAY_19", "High")
        recurse(tree.children_left[node], path)                              # left: NOT this partition -- dropped
        recurse(tree.children_right[node], path + [(feature, partition)])    # right: IS this partition -- kept

    recurse(0, [])

    seen_condition_sets = set()
    candidates = []
    for conditions_raw in leaf_paths:
        if not conditions_raw:
            continue  # a leaf reached entirely through negative branches -- no positive rule to extract
        conditions = [{"feature": f, "partition": p} for f, p in conditions_raw]
        dedup_key = tuple(sorted((c["feature"], c["partition"]) for c in conditions))
        if dedup_key in seen_condition_sets:
            continue  # multiple leaves can decode to the same condition set once negative branches drop out
        seen_condition_sets.add(dedup_key)

        mask = metrics.evaluate_rule_mask(discretised, conditions)
        coverage = float(mask.mean())
        if mask.sum() == 0 or coverage < MIN_COVERAGE:
            continue

        predicted_class = discretised.loc[mask, class_col].mode().iat[0]  # majority vote among covered rows,
                                                                            # same convention as nsga.py's _score_genotype
        rule_preds, model_preds, _ = metrics.rule_class_preds(discretised, conditions, predicted_class, class_col=class_col)
        fidelity = metrics.fidelity_f1(rule_preds, model_preds)

        candidates.append({
            "conditions": conditions,
            "genes": resolve_genes(conditions, feature_map),
            "predicted_class": predicted_class,
            "length": metrics.comprehensibility({"conditions": conditions}),
            "fidelity_f1": round(float(fidelity), 4),
            "coverage": round(coverage, 4),
            "run_ids": [1],  # CART is deterministic given a fixed random_state -- one "run"
            "non_dominated": True,  # not a Pareto-front method; kept for schema compatibility (see §5)
        })

    candidates.sort(key=lambda r: (-r["fidelity_f1"], r["length"]))
    return candidates


for condition, class_col in CLASS_COL.items():
    preview = extract_cart_rules(condition, class_col, seed=cfg.seed)
    print(f"{condition}: {len(preview)} rules survived the {MIN_COVERAGE:.0%} coverage floor")


baseline: 9 rules survived the 2% coverage floor
sam: 12 rules survived the 2% coverage floor


## 4. Sanity check against the planted ground truth

Same check as notebooks 02/03/05: do the highest-fidelity CART rules actually mention the
planted `TRUE_SIGNAL_GENES`/`TRUE_SIGNAL_PATHWAYS`? If CART — a completely independent
extraction method from GA/NSGA-II's chromosome-based search — recovers the same planted
signal, that's a meaningful cross-method validation, not just a code-runs check.


In [5]:
ground_truth = json.load(open(FIXTURE_DIR / "synthetic_ground_truth_meta.json"))
true_features = set(ground_truth["true_signal_genes"].keys()) | set(ground_truth["true_signal_pathways"].keys())

for condition, class_col in CLASS_COL.items():
    rules = extract_cart_rules(condition, class_col, seed=cfg.seed)
    top5 = rules[:5]
    touched = set()
    for r in top5:
        touched.update(c["feature"] for c in r["conditions"])
    overlap = touched & true_features
    print(f"{condition}: top-5 CART rules touch {sorted(touched)}")
    print(f"  -> overlap with planted signal: {sorted(overlap)}")


baseline: top-5 CART rules touch ['GENE_06', 'PATHWAY_10', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
  -> overlap with planted signal: ['GENE_06', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']


sam: top-5 CART rules touch ['GENE_06', 'GENE_11', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
  -> overlap with planted signal: ['GENE_06', 'GENE_11', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']


## 5. Write schema-valid rule pools

Written to `tests/fixtures/synthetic/rules/cart_{baseline,sam}.json` — same directory as the
fabricated `ga_*`/`nsga_*` pools, since these are also built from synthetic fixture data, not
real predictions yet. Unlike `ga_*`/`nsga_*`, these CART rules **are** the real extraction
logic (not a stand-in for someone else's not-yet-built branch) — only the input data is
synthetic. `schema.save_rule_pool` validates before writing, exercising the `"cart"` fix from
§1 for real.


In [6]:
OUT_DIR = RULES_DIR
config_hash_val = cfg.config_hash

cart_pools = {}
for condition, class_col in CLASS_COL.items():
    rules = extract_cart_rules(condition, class_col, seed=cfg.seed)
    for i, r in enumerate(rules, start=1):
        r["id"] = f"cart_{condition}_{i:04d}"

    payload = {
        "meta": {
            "extractor": "cart",
            "condition": condition,
            "runs": 1,
            "config_hash": config_hash_val,
        },
        "rules": rules,
    }
    out_path = OUT_DIR / f"cart_{condition}.json"
    schema.save_rule_pool(payload, out_path, feature_map=feature_map)  # validates, then writes
    cart_pools[f"cart_{condition}"] = payload
    print(f"cart_{condition}: {len(rules)} rules -> {out_path}")


cart_baseline: 9 rules -> ../tests/fixtures/synthetic/rules/cart_baseline.json


cart_sam: 12 rules -> ../tests/fixtures/synthetic/rules/cart_sam.json


## 6. Re-running notebook 03's unified PROMETHEE with CART folded in

Per CONTRACTS.md §4.6.4, CART's rules must go through the **same** unified PROMETHEE run as
GA/NSGA-II, not a separate one. Rather than duplicating notebook 03's logic here, this section
demonstrates the fold-in against the six-pool combined set (`ga_*`, `nsga_*`, `cart_*`) so the
GA/NSGA-II vs CART comparison is possible — notebook 03 itself should be re-run (or extended)
to make this the standing combined ranking once CART is confirmed as a permanent fixture
rather than the time-permitting extension it currently is (Part F2).


In [7]:
from src.common.schema import load_rule_pool
from pymcdm.methods import PROMETHEE_II

ALL_POOL_NAMES = ["ga_baseline", "ga_sam", "nsga_baseline", "nsga_sam", "cart_baseline", "cart_sam"]
all_pools = {name: load_rule_pool(RULES_DIR / f"{name}.json", feature_map=feature_map) for name in ALL_POOL_NAMES}


def rule_stability(rule: dict, condition: str) -> float:
    class_col = CLASS_COL[condition]
    rule_preds_by_subgroup, model_preds_by_subgroup = {}, {}
    for cancer_type, subgroup_df in discretised.groupby("cancer_type"):
        mask = metrics.evaluate_rule_mask(subgroup_df, rule["conditions"])
        if mask.sum() == 0:
            continue
        rp, mp, _ = metrics.rule_class_preds(subgroup_df, rule["conditions"], rule["predicted_class"], class_col=class_col)
        rule_preds_by_subgroup[cancer_type] = rp
        model_preds_by_subgroup[cancer_type] = mp
    if len(rule_preds_by_subgroup) < 2:
        return np.nan
    return metrics.stability(rule_preds_by_subgroup, model_preds_by_subgroup)


rows = []
for name, pool in all_pools.items():
    extractor, condition = pool["meta"]["extractor"], pool["meta"]["condition"]
    for r in pool["rules"]:
        rows.append({
            "id": r["id"], "extractor": extractor, "condition": condition,
            "fidelity_f1": r["fidelity_f1"], "length": r["length"], "coverage": r["coverage"],
            "stability": rule_stability(r, condition),
        })
combined_with_cart = pd.DataFrame(rows)
print(f"{len(combined_with_cart)} rules across {combined_with_cart['extractor'].nunique()} extractors "
      f"(vs 60 across 2 extractors in notebook 03)")

# same criteria/weights/preference function as notebook 03 -- provisional, see that notebook's §3
CRITERIA = ["fidelity_f1", "length", "stability"]
WEIGHTS = np.array([0.5, 0.2, 0.3])
TYPES = np.array([1, -1, -1])

matrix = combined_with_cart[CRITERIA].to_numpy(dtype=float)
p = matrix.max(axis=0) - matrix.min(axis=0)
q = np.zeros(len(CRITERIA))
promethee = PROMETHEE_II("vshape", p=p, q=q)
combined_with_cart["phi"] = promethee(matrix, WEIGHTS, TYPES)
combined_with_cart = combined_with_cart.sort_values("phi", ascending=False).reset_index(drop=True)

print()
print("top 10 across all six pools (GA, NSGA-II, CART x baseline, SAM):")
combined_with_cart.head(10)[["id", "extractor", "condition", "fidelity_f1", "length", "stability", "phi"]]


81 rules across 3 extractors (vs 60 across 2 extractors in notebook 03)

top 10 across all six pools (GA, NSGA-II, CART x baseline, SAM):


,id,extractor,condition,fidelity_f1,length,stability,phi
0,ga_baseline_0001,ga,baseline,1.0000,2,0.000000,0.548027
1,ga_sam_0001,ga,sam,1.0000,2,0.000000,0.548027
2,nsga_baseline_0001,nsga,baseline,1.0000,2,0.000000,0.548027
3,nsga_sam_0001,nsga,sam,1.0000,2,0.000000,0.548027
4,ga_sam_0002,ga,sam,1.0000,3,0.000000,0.497402
5,ga_baseline_0002,ga,baseline,1.0000,4,0.000000,0.446777
6,ga_sam_0009,ga,sam,0.4333,1,0.001338,0.166168
7,nsga_sam_0014,nsga,sam,0.4333,1,0.001338,0.166168
8,cart_baseline_0005,cart,baseline,0.4125,1,0.000570,0.151618
9,cart_sam_0009,cart,sam,0.3855,1,0.004749,0.124346


In [8]:
from scipy.stats import mannwhitneyu

evolutionary_phi = combined_with_cart.loc[combined_with_cart["extractor"].isin(["ga", "nsga"]), "phi"]
cart_phi = combined_with_cart.loc[combined_with_cart["extractor"] == "cart", "phi"]

u_stat, p_value = mannwhitneyu(evolutionary_phi, cart_phi, alternative="two-sided")
print(f"evolutionary (GA+NSGA-II, n={len(evolutionary_phi)}) vs CART (n={len(cart_phi)}): "
      f"U={u_stat:.1f}, p={p_value:.4f}")
print(f"medians: evolutionary={evolutionary_phi.median():.3f}, CART={cart_phi.median():.3f}")


evolutionary (GA+NSGA-II, n=60) vs CART (n=21): U=442.0, p=0.0433
medians: evolutionary=-0.048, CART=0.058


## 7. Save outputs


In [9]:
REPORT_DIR = FIXTURE_DIR / "reports"
combined_with_cart.to_csv(REPORT_DIR / "promethee_ranked_rules_with_cart.csv", index=False)
print(f"wrote {len(combined_with_cart)}-row combined ranking (incl. CART) -> "
      f"{REPORT_DIR / 'promethee_ranked_rules_with_cart.csv'}")


wrote 81-row combined ranking (incl. CART) -> ../tests/fixtures/synthetic/reports/promethee_ranked_rules_with_cart.csv


## Known gaps, to address when real data arrives

- **Rule-extraction convention (dropping negative branches) is a modelling choice**, not the
  only valid one — an alternative would allow negated conditions (`feature != partition`),
  but that's not representable in the current §0.4 schema (`VALID_PARTITIONS` are the three
  positive labels only). Flag to the team if negated conditions are wanted; would need a
  schema change beyond the `VALID_EXTRACTORS` addition in §1.
- **`max_depth=4` / `min_samples_leaf=5` are arbitrary starting points**, not tuned or
  justified beyond "keeps rules short." Worth a small sensitivity check once real data
  arrives (does rule quality change meaningfully across depth 3–6?).
- **`schema.py`'s `VALID_EXTRACTORS` change needs team announcement** (§1) — technically done
  in this session's copy of the repo, needs to land as a real PR against the shared branch.
- Notebook 03 itself should be updated to load six pools instead of four once CART's status
  (required vs time-permitting, Part F2) is settled with supervisors.

## Next up

With PROMETHEE, statistical testing, GSEA, and CART all wired against synthetic fixtures and
running end to end, the evaluation layer's code is ready for real inputs. Remaining work is
data, not code: swap `tests/fixtures/synthetic/` for `artifacts/data/` + `artifacts/rules/*`
once Angelique's, Bahiya's, Melisa's, and Sakhile's real branches land, then re-run 03→06
against real output — per the orientation doc's Phase 3 target (~18 September).
